# Demand Forecasting
## Using Histogram Binning with Gradient Boosting Regressor

---

In [24]:
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sys


In [25]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import combine_raw_data

PROCESSED_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "combined_demand.csv"

combine_raw_data()
df = pd.read_csv(PROCESSED_CSV_PATH, parse_dates=["SETTLEMENT_DATE"])

c:\Users\roodruh\Desktop\repos\neso-demand-forecasting\src\data_processing.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"])


Successfully combined 5 files into: C:\Users\roodruh\Desktop\repos\neso-demand-forecasting\data\processed\combined_demand.csv


In [26]:
df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"])

df = df.sort_values(by = ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]).reset_index(drop=True)

uk_holidays = holidays.UK(years=[2021, 2022, 2023, 2024, 2025])

df["IS_HOLIDAY"] = (
    df["SETTLEMENT_DATE"].dt.date.isin(uk_holidays).astype(int)
)

df["ND_LAG_48"] = df["ND"].shift(48)
df["ND_LAG_336"] = df["ND"].shift(336)
df["ND_AVG_7D"] = (
    df["ND"].shift(1).rolling(window=336).mean()
)

df = df.dropna().reset_index(drop=True)

In [27]:
non_feature_cols = ["ND", "SETTLEMENT_DATE"]

split_date = "2025-01-01"

train_mask = df["SETTLEMENT_DATE"] < split_date
test_mask = df["SETTLEMENT_DATE"] >= split_date

X_train = df.loc[train_mask].drop(columns=non_feature_cols)
y_train = df.loc[train_mask, "ND"]

X_test = df.loc[test_mask].drop(columns=non_feature_cols)
y_test = df.loc[test_mask, "ND"]

In [30]:
model = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.03,
    max_leaf_nodes=63,
    min_samples_leaf=50,
    max_bins=255,
    l2_regularization=1.0,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [33]:
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
r2 = r2_score(y_test, preds)

print(f"MAE:   {mae:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAPE:  {mape:.2f}%")
print(f"R²:    {r2:.4f}")

MAE:   181.95
RMSE:  242.81
MAPE:  0.73%
R²:    0.9985
